# 22. Varian CNN (modifikasi penulis) pada data Palm bersih
**Penelitian:** Deteksi anemia dari citra pucat (telapak tangan, kuku, konjungtiva) dengan data pooling multi-situs.

**Tujuan:** menguji tiga varian yang ditetapkan sebelum melihat hasil uji dan dilaporkan seluruhnya, untuk menjawab apakah kolaps AlexNet pada Notebook 21 berkaitan dengan skala regularisasi L2 (Keras l2 tidak dibagi jumlah sampel, sedangkan alpha pada MLP scikit-learn dibagi 2n) dan pengoptimal. Varian A: L2 skala sklearn, SGD, 10 epoch. Varian B: A ditambah momentum 0,9 dan early stopping pada loss validasi. Varian MobileNetV2 dengan early stopping pada loss validasi. Pemilihan epoch hanya memakai data validasi; data uji dibaca sekali. Baris baseline Notebook 21 (konfigurasi [1]) tidak digantikan.

**Masukan:** Dataset (Mendeley Data): Palm doi:10.17632/ccr8cm22vz.1, Nail doi:10.17632/2xx4j3kjg2.1, Conjunctiva doi:10.17632/nt7r8hv2pz.1. Letakkan dalam folder `Dataset_Multisite/{Palm,Nail,Conjunctiva}`.

**Keluaran (folder `hasil_revisi`):** `varian_cnn_palm_data_bersih.csv`, `riwayat_varian_cnn.csv`, `kurva_varian_cnn.png`.

**Kaitan dengan naskah:** Tabel 2 dan Lampiran A1 (baris varian, berlabel modifikasi penulis).

**Cara menjalankan:** Colab, Runtime > Run all. CPU sudah cukup; GPU tidak diperlukan. Seed acak = 42 dan urutan file diurutkan (`sorted`) sehingga hasil dapat direproduksi pada versi pustaka yang dicetak di sel kedua.

In [ ]:
!pip install lightgbm -q

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

In [ ]:
# ===== FUNGSI BERSAMA (identik di keempat notebook) =====
import os, re, hashlib, warnings
import numpy as np, pandas as pd, cv2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score
from lightgbm import LGBMClassifier
warnings.filterwarnings('ignore')

# ---------- KONFIGURASI (ubah hanya bagian ini) ----------
SEED = 42
BASE = '/content/drive/MyDrive/Dataset_Multisite'   # folder berisi subfolder Palm/, Nail/, Conjunctiva/
OUT  = '/content/drive/MyDrive/hasil_revisi_v2'          # folder keluaran (CSV)
SITES = {'Palm': 0, 'Nail': 1, 'Conjunctiva': 2}
# Filter piksel memeriksa kanal L, R, dan G. Kanal B ikut difilter jika True
# (Appiahene et al. 2023 menuliskan kondisi min < R=G=B < max).
FILTER_INCLUDE_B = False
# True: buang juga citra bernama varian dari citra lain (akhiran FV/RA atau spasi+angka) yang citra dasarnya ada.
BUANG_VARIAN = True
os.makedirs(OUT, exist_ok=True)

# ---------- DAFTAR FILE ----------
def parse_label(name):
    """Label dari awalan nama file (Anemic / Non-Anemic); nama yang mengandung tanda kurung (salinan bernomor) dan awalan tidak dikenali dilewati."""
    if '(' in name: return None
    c = name.lower().replace(' ', '')
    if c.startswith('non-anemic'): return 0
    if c.startswith('anemic'): return 1
    return None

def list_used_files():
    rows = []
    for site, sid in SITES.items():
        d = os.path.join(BASE, site)
        for n in sorted(os.listdir(d)):          # sorted -> urutan deterministik
            lab = parse_label(n)
            if lab is not None:
                rows.append(dict(site=site, site_id=sid, file=n, path=os.path.join(d, n), label=lab))
    return pd.DataFrame(rows)

def md5(path):
    with open(path, 'rb') as f: return hashlib.md5(f.read()).hexdigest()

# ---------- EKSTRAKSI FITUR ----------
def feat_original(img, lo=50, hi=200):
    """Fitur [mean a*, mean b*, mean G] dengan filter piksel. Return (fitur, mask_kosong)."""
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    L, a, b = cv2.split(lab); R, G, B = cv2.split(rgb)
    m = (L > lo) & (L < hi) & (R > lo) & (R < hi) & (G > lo) & (G < hi)
    if FILTER_INCLUDE_B: m &= (B > lo) & (B < hi)
    if m.any(): return [a[m].mean(), b[m].mean(), G[m].mean()], False
    return [0.0, 0.0, 0.0], True

def feat_otsu(img):
    """Fitur dari piksel foreground Otsu (grayscale). Tanpa filter L/R/G. Return (fitur, mask_kosong)."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, mk = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = mk > 0
    if not m.any(): return [0.0, 0.0, 0.0], True
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    _, a, b = cv2.split(lab); G = rgb[:, :, 1]
    return [a[m].mean(), b[m].mean(), G[m].mean()], False

# ---------- PEMBAGIAN DATA & EVALUASI ----------
def split_idx(y, scheme, seed=SEED):
    """Return (train, val, test) berupa indeks. Skema: '80:20', '80:10:10', '70:10:20'."""
    idx = np.arange(len(y))
    if scheme == '80:20':
        tr, te = train_test_split(idx, test_size=0.20, stratify=y, random_state=seed)
        return tr, None, te
    if scheme == '80:10:10':
        tr, tmp = train_test_split(idx, test_size=0.20, stratify=y, random_state=seed)
        va, te = train_test_split(tmp, test_size=0.50, stratify=y[tmp], random_state=seed)
        return tr, va, te
    if scheme == '70:10:20':
        trva, te = train_test_split(idx, test_size=0.20, stratify=y, random_state=seed)
        tr, va = train_test_split(trva, test_size=0.125, stratify=y[trva], random_state=seed)
        return tr, va, te
    raise ValueError(scheme)

def metrics_from(yte, pred, proba):
    tn, fp, fn, tp = confusion_matrix(yte, pred, labels=[0, 1]).ravel()
    assert tn + fp + fn + tp == len(yte), 'TP+TN+FP+FN harus sama dengan jumlah data uji'
    two = len(set(yte)) == 2
    return dict(n_test=len(yte), n_test_anemic=int((yte == 1).sum()), TP=int(tp), TN=int(tn), FP=int(fp), FN=int(fn),
                acc=100*(tp+tn)/len(yte),
                sens=100*tp/(tp+fn) if tp+fn else np.nan,
                spec=100*tn/(tn+fp) if tn+fp else np.nan,
                f1=100*f1_score(yte, pred, zero_division=0),
                auc=100*roc_auc_score(yte, proba) if two else np.nan,
                baseline_mayoritas=100*max((yte == 1).mean(), (yte == 0).mean()))

def fit_lgbm(Xtr, ytr):
    sc = StandardScaler().fit(Xtr)
    m = LGBMClassifier(n_estimators=100, random_state=SEED, verbose=-1).fit(sc.transform(Xtr), ytr)
    return sc, m

def predict_lgbm(sc, m, X):
    proba = m.predict_proba(sc.transform(X))[:, 1]
    return (proba >= 0.5).astype(int), proba

def tandai_dibuang(df):
    """Array boolean (urut baris): True bila baris dibuang pada pembersihan = salinan identik MD5 (satu dipertahankan),
    kedua file pada pasangan identik berlabel berbeda, dan (bila BUANG_VARIAN) citra varian FV/RA. df: indeks 0..n-1 dan kolom md5."""
    konflik = df.groupby('md5').label.transform('nunique') > 1
    dibuang = np.array(df.duplicated('md5', keep='first') | konflik, dtype=bool)   # salinan yang dapat diubah
    if BUANG_VARIAN:
        posisi = np.flatnonzero(~dibuang)
        for a, b in pasangan_varian(df.iloc[posisi].reset_index(drop=True)): dibuang[posisi[a]] = True
    return dibuang

def bersihkan(df):
    """Data bersih untuk semua eksperimen (satu definisi tunggal)."""
    df = df.reset_index(drop=True).copy(); df['md5'] = df['path'].map(md5)
    return df[~tandai_dibuang(df)].reset_index(drop=True)

def cek_kebocoran(md5s, tr, va, te, wajib=True):
    """Pemeriksaan kebocoran: indeks latih/validasi/uji harus saling lepas dan tidak ada citra identik (MD5) lintas partisi.
    Return jumlah citra identik lintas partisi, mis. {'latih-uji': 0}. wajib=True -> hentikan program bila ada pelanggaran."""
    parts = {'latih': set(tr), 'uji': set(te)}
    if va is not None: parts['val'] = set(va)
    nm = sorted(parts)
    for i, a in enumerate(nm):
        for b in nm[i + 1:]:
            assert not (parts[a] & parts[b]), f'Indeks tumpang tindih: {a}-{b}'
    m = {k: set(np.asarray(md5s)[sorted(v)]) for k, v in parts.items()}
    bocor = {f'{a}-{b}': len(m[a] & m[b]) for i, a in enumerate(nm) for b in nm[i + 1:]}
    if wajib: assert all(v == 0 for v in bocor.values()), f'Citra identik lintas partisi: {bocor}'
    return bocor

def pasangan_varian(df):
    """Pasangan (indeks varian, indeks citra dasar) pada df (indeks posisi 0..n-1): nama berakhiran FV/RA atau spasi+angka
    yang nama dasarnya ada pada situs yang sama (mis. Anemic-001FV dengan Anemic-001)."""
    stem = df.file.str.replace(r'\.[A-Za-z]+$', '', regex=True)
    base = stem.str.replace(r'\s\d+$', '', regex=True).str.replace(r'(FV|RA)$', '', regex=True)
    key = {(s, t): i for i, (s, t) in enumerate(zip(df.site, stem))}
    return [(i, key[(s, b)]) for i, (s, t, b) in enumerate(zip(df.site, stem, base)) if b != t and (s, b) in key]

def cek_varian(pasangan, tr, va, te):
    """Jumlah pasangan varian-dasar yang terpisah di partisi berbeda (informasi; idealnya 0)."""
    part = {}
    for ids, nm in [(tr, 'latih'), (va, 'val'), (te, 'uji')]:
        if ids is not None:
            for i in ids: part[i] = nm
    return sum(1 for a, b in pasangan if a in part and b in part and part[a] != part[b])

In [ ]:
import sys, numpy, pandas, cv2, sklearn, lightgbm
print('Python', sys.version.split()[0], '| numpy', numpy.__version__, '| pandas', pandas.__version__,
      '| OpenCV', cv2.__version__, '| scikit-learn', sklearn.__version__, '| LightGBM', lightgbm.__version__, '| SEED', SEED)

In [ ]:
# ===== VARIAN CNN (MODIFIKASI PENULIS) PADA DATA PALM BERSIH: tidak menggantikan baseline Notebook 21 =====
import tensorflow as tf
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models, optimizers, regularizers

SCHEMES = ['70:10:20', '80:10:10']
IMG, BATCH = 224, 32
EPOCHS_MAKS, SABAR = 40, 5             # batas epoch dan kesabaran early stopping (loss validasi)
MN_PRETRAINED = 'imagenet'
DETERMINISTIK = True
tf.keras.utils.set_random_seed(SEED)
if DETERMINISTIK:
    try: tf.config.experimental.enable_op_determinism()
    except Exception as e: print('Determinisme operasi tidak aktif:', e)
print('GPU:', tf.config.list_physical_devices('GPU'))

palm = bersihkan(list_used_files()); palm = palm[palm.site == 'Palm'].reset_index(drop=True)
y = palm.label.to_numpy()
def prep(path):
    im = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    return cv2.resize(im, (IMG, IMG), interpolation=cv2.INTER_AREA)
IM = np.stack([prep(p) for p in palm.path]).astype('uint8')
print('Citra Palm bersih:', len(y), '| anemic:', int(y.sum()), '| non-anemic:', int((y == 0).sum()), '| bentuk:', IM.shape)
def versi(x): return [x, np.rot90(x, -1), np.rot90(x, 1), x[::-1], x[:, ::-1]]

def build_alexnet(l2=1e-4, momentum=0.0):
    reg = regularizers.l2(l2)
    m = models.Sequential([
        layers.Input((IMG, IMG, 3)), layers.Rescaling(1. / 255),
        layers.Conv2D(96, (11, 11), strides=4, activation='relu', kernel_regularizer=reg), layers.MaxPooling2D((3, 3), strides=2),
        layers.Conv2D(256, (5, 5), padding='same', activation='relu', kernel_regularizer=reg), layers.MaxPooling2D((3, 3), strides=2),
        layers.Conv2D(384, (3, 3), padding='same', activation='relu', kernel_regularizer=reg),
        layers.Conv2D(384, (3, 3), padding='same', activation='relu', kernel_regularizer=reg),
        layers.Conv2D(256, (3, 3), padding='same', activation='relu', kernel_regularizer=reg), layers.MaxPooling2D((3, 3), strides=2),
        layers.Flatten(), layers.Dense(128, activation='relu', kernel_regularizer=reg), layers.Dropout(0.5),
        layers.Dense(128, activation='relu', kernel_regularizer=reg), layers.Dropout(0.5), layers.Dense(1, activation='sigmoid')])
    m.compile(optimizer=optimizers.SGD(learning_rate=0.01, momentum=momentum), loss='binary_crossentropy', metrics=['accuracy'])
    return m

def build_mobilenet():
    base = tf.keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights=MN_PRETRAINED)
    base.trainable = False
    inp = layers.Input((IMG, IMG, 3)); x = layers.Rescaling(1. / 127.5, offset=-1)(inp)
    x = layers.Dropout(0.2)(layers.GlobalAveragePooling2D()(base(x, training=False)))
    m = models.Model(inp, layers.Dense(1, activation='sigmoid')(x))
    m.compile(optimizer=optimizers.Adam(learning_rate=1e-4), loss='binary_crossentropy', metrics=['accuracy']); return m

# Tiga varian yang ditetapkan SEBELUM melihat hasil uji; semua dilaporkan (tidak ada yang dibuang).
# L2 skala sklearn: alpha/(2n) dengan n = jumlah baris latih; Keras l2() tidak membagi dengan n.
VARIAN = {
 'AlexNet-A (L2 skala sklearn, SGD, 10 epoch)': dict(builder=lambda n: build_alexnet(l2=1e-4 / (2 * n)), epochs=10, es=False),
 'AlexNet-B (L2 skala sklearn, SGD momentum 0,9, early stopping)': dict(builder=lambda n: build_alexnet(l2=1e-4 / (2 * n), momentum=0.9), epochs=EPOCHS_MAKS, es=True),
 'MobileNetV2 (transfer learning, early stopping)': dict(builder=lambda n: build_mobilenet(), epochs=EPOCHS_MAKS, es=True)}

rows, riwayat = [], []
for scheme in SCHEMES:
    tr, va, te = split_idx(y, scheme)
    bocor = cek_kebocoran(palm.md5.to_numpy(), tr, va, te)
    vlin = cek_varian(pasangan_varian(palm), tr, va, te)
    Xtr = np.stack([np.ascontiguousarray(v) for i in tr for v in versi(IM[i])]); ytr = np.repeat(y[tr], 5); grp = np.repeat(tr, 5)
    assert not (set(grp) & set(te)) and not (set(grp) & set(va)), 'Salinan augmentasi citra uji/validasi masuk ke data latih'
    print(f'\nSkema {scheme}: pemeriksaan kebocoran lolos {bocor}, pasangan varian lintas partisi={vlin}; citra latih={len(tr)} (baris={len(ytr)}), val={len(va)}, uji={len(te)}')
    for nama, cfg in VARIAN.items():
        tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(SEED)
        m = cfg['builder'](len(ytr))
        cb = [tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=SABAR, restore_best_weights=True)] if cfg['es'] else []
        h = m.fit(Xtr, ytr, batch_size=BATCH, epochs=cfg['epochs'], validation_data=(IM[va], y[va]), shuffle=True, callbacks=cb, verbose=0)
        pb = m.predict(IM[te], batch_size=BATCH, verbose=0).ravel(); vb = m.predict(IM[va], batch_size=BATCH, verbose=0).ravel()
        pr = (pb >= 0.5).astype(int)
        r = metrics_from(y[te], pr, pb); v = metrics_from(y[va], (vb >= 0.5).astype(int), vb)
        prec = 100 * r['TP'] / (r['TP'] + r['FP']) if (r['TP'] + r['FP']) else np.nan
        vl = h.history['val_loss']
        rows.append(dict(skema=scheme, algoritma=nama, catatan='modifikasi penulis', n_latih_baris=len(ytr), n_val=len(va), **r, precision=prec,
                         pct_prediksi_anemic=100 * pr.mean(), val_acc=v['acc'], val_auc=v['auc'],
                         epoch_berhenti=len(vl), epoch_terbaik=int(np.argmin(vl)) + 1 if cfg['es'] else len(vl)))
        for e in range(len(vl)):
            riwayat.append(dict(skema=scheme, algoritma=nama, epoch=e + 1, **{kk: vv[e] for kk, vv in h.history.items()}))
        print(f'  {nama}: acc={r["acc"]:.2f} auc={r["auc"]:.2f} sens={r["sens"]:.2f} spec={r["spec"]:.2f} | TP={r["TP"]} TN={r["TN"]} FP={r["FP"]} FN={r["FN"]} | diprediksi anemic={100*pr.mean():.1f}% | epoch={len(vl)}')

hasil = pd.DataFrame(rows)
kolom = ['skema','algoritma','catatan','n_latih_baris','n_val','n_test','n_test_anemic','TP','TN','FP','FN','acc','precision','sens','spec','f1','auc',
         'val_acc','val_auc','baseline_mayoritas','pct_prediksi_anemic','epoch_berhenti','epoch_terbaik']
hasil = hasil[kolom].round(2); hasil.to_csv(os.path.join(OUT, 'varian_cnn_palm_data_bersih.csv'), index=False)
rw = pd.DataFrame(riwayat).round(4); rw.to_csv(os.path.join(OUT, 'riwayat_varian_cnn.csv'), index=False)
print('\n', hasil[['skema','algoritma','acc','precision','sens','spec','f1','auc','val_acc','val_auc','baseline_mayoritas','pct_prediksi_anemic','epoch_terbaik']].to_string(index=False))

fig, ax = plt.subplots(len(SCHEMES), len(VARIAN), figsize=(5 * len(VARIAN), 4 * len(SCHEMES)), squeeze=False)
for i, scheme in enumerate(SCHEMES):
    for j, nama in enumerate(VARIAN):
        d = rw[(rw.skema == scheme) & (rw.algoritma == nama)]
        ax[i][j].plot(d.epoch, d['loss'], label='loss latih'); ax[i][j].plot(d.epoch, d['val_loss'], label='loss validasi')
        ax[i][j].set_title(f'{nama[:34]}\n({scheme})', fontsize=9); ax[i][j].set_xlabel('epoch'); ax[i][j].legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT, 'kurva_varian_cnn.png'), dpi=150)